# 01 · My Ray cluster, from a notebook

**Who this is for:** a JupyterHub user whose Lab has the Bifrost sidebar. You do not need
`kubectl`, a Bifrost token, or to know where the API lives — the extension already knows,
and it acts as *you* (your project, your quota, your cluster only).

This notebook:

1. shows the profiles the administrator lets your project use,
2. starts a cluster from one (or picks up the one you already have running),
3. submits a Ray job to it through the Ray Jobs API and reads its logs,
4. leaves the cluster running for `02-checkmaite-capability.ipynb` and `03-stress-ramp.ipynb`.

Environment knobs (all optional): `BIFROST_PROFILE` (default `checkmaite`), `BIFROST_CLUSTER_ID`
(reuse a specific cluster instead of the one running one).

In [ ]:
# The Bifrost sidebar talks to a small server extension inside this very
# JupyterLab (`/user/<you>/bifrost/*`). A notebook can call the same routes
# with the server's own hub token, so what happens here is exactly what a
# click in the sidebar does: same identity, same project, same NetworkPolicy.
import os, time, json, requests

SERVER = os.environ["JUPYTERHUB_SERVICE_URL"]          # http://0.0.0.0:8888/user/<you>/
USER = os.environ.get("JUPYTERHUB_USER", "me")
_HDR = {"Authorization": f"token {os.environ['JUPYTERHUB_API_TOKEN']}"}

def ext(method, path, body=None, **kw):
    """Call an extension route; returns (status, json-or-text)."""
    r = requests.request(method, SERVER + "bifrost/" + path, headers=_HDR, json=body, timeout=60, **kw)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text

def my_running_cluster():
    """The cluster these notebooks share: BIFROST_CLUSTER_ID if set, else the one running cluster."""
    want = os.environ.get("BIFROST_CLUSTER_ID")
    status, view = ext("GET", "clusters")
    assert status == 200 and view.get("configured", True), f"extension not configured: {status} {view}"
    running = [c for c in view["clusters"] if c["state"] == "running"]
    if want:
        return next((c for c in running if c["id"] == want), None)
    return running[0] if len(running) == 1 else None

def wait_running(cluster_id, timeout=900):
    t0 = time.time()
    while time.time() - t0 < timeout:
        status, view = ext("GET", "clusters")
        state = next((c["state"] for c in view["clusters"] if c["id"] == cluster_id), "gone")
        print(f"{time.time()-t0:5.0f}s  {cluster_id}: {state}", flush=True)
        if state == "running":
            return
        if state in ("failed", "terminated", "gone"):
            raise RuntimeError(f"cluster {cluster_id} went {state}")
        time.sleep(10)
    raise TimeoutError(f"cluster {cluster_id} not running after {timeout}s")

print("notebook user:", USER, "| server:", SERVER)

## What may I start?

Profiles are the administrator's cluster shapes: image, head size, worker group with min/max
replicas, storage the cluster mounts. You pick a name; Bifrost fills in the rest and refuses
anything outside it.

In [ ]:
status, view = ext("GET", "profiles")
assert status == 200, (status, view)
status, project = ext("GET", "project")
print("project this notebook lands in:", project.get("project"), "| choices:", project.get("projects"))
for p in view["profiles"]:
    print(f'- {p["name"]:12s} ' + "  ".join(f"{k}={v}" for k, v in p.items() if k != "name"))

## Start one (or reuse mine)

The POST carries nothing but the profile name — that is the whole point. The cluster is
yours: only your notebook pod can reach its head, and only you (and your project's operators)
see it in the list.

In [ ]:
PROFILE = os.environ.get("BIFROST_PROFILE", "checkmaite")

cluster = my_running_cluster()
if cluster is None:
    status, created = ext("POST", "clusters", {"profile": PROFILE})
    assert status == 200, f"start refused: {status} {created}"
    print("started", created)
    CLUSTER_ID = created["id"]
    wait_running(CLUSTER_ID)
else:
    CLUSTER_ID = cluster["id"]
    print("reusing running cluster", CLUSTER_ID)

status, addr = ext("GET", f"clusters/{CLUSTER_ID}/address")
print(json.dumps(addr, indent=1))

## A first job

`bifrost_jupyter.connect` hands back Ray's own `JobSubmissionClient`, pointed at the head
service inside the cluster. Nothing here is Bifrost-specific past that line — it is the Ray
Jobs API, the way the Ray docs describe it.

In [ ]:
from bifrost_jupyter import connect

client = connect(CLUSTER_ID)
sub = client.submit_job(
    entrypoint='python -c "import ray, os; ray.init(); print(\'hello from\', os.environ[\'WHO\'], \'| nodes:\', len(ray.nodes()), \'| cpus:\', ray.cluster_resources().get(\'CPU\'))"',
    runtime_env={"env_vars": {"WHO": USER}},
)
print("submitted", sub)
while (st := client.get_job_status(sub)) not in ("SUCCEEDED", "FAILED", "STOPPED"):
    time.sleep(3)
print("status:", st)
print(client.get_job_logs(sub))
assert st == "SUCCEEDED"

## Where the other notebooks pick up

The cluster stays running (it will be reaped by the profile's idle timeout / TTL if you
forget it, and `04-cleanup.ipynb` deletes it on purpose). The advanced path — the Ray Client
at `ray://…:10001`, which lets a library like checkmaite drive the cluster directly — is
what notebook 02 uses.

In [ ]:
print("cluster id:", CLUSTER_ID)
print("ray client :", addr["ray_client_address"])